# 09 · Experimentación, MLOps y Decision Memory
Cierra el ciclo: **predicción → acción → resultado → aprendizaje**. Un modelo que sólo predice no es un sistema de decisión.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel
install_feature_mart(); panel=load_monthly_panel()


## Model card mínimo
Cada ejecución debe registrar: fecha de corte, horizonte, target, features, proyectos, ventana de entrenamiento, holdout, MAE/RMSE, champion, baseline, restricciones y calidad del ledger.

In [ ]:
model_card=pd.DataFrame([
 {'campo':'target','valor':'movimiento neto próximos N meses / minutas próximos N meses'},
 {'campo':'unidad','valor':'proyecto-mes'},
 {'campo':'validación','valor':'holdout temporal de meses recientes'},
 {'campo':'leakage','valor':'excluir precios actuales de entrenamiento histórico por defecto'},
 {'campo':'stockout','valor':'derivado de demanda neta, no de minutas'},
 {'campo':'causalidad','valor':'no inferida por ML; requiere experimento o diseño econométrico'}
])
model_card

## Drift de datos
Comparar la distribución de los últimos 6 meses contra la historia anterior. PSI/KS pueden añadirse; aquí usamos cambios de media estandarizada como detector simple y auditable.

In [ ]:
features=['stock_inicio_observado','movimiento_neto_mes','ventas_minutas_mes','caidas_mes','absorcion_neta_mes']
cut=panel['periodo_mes'].max()-pd.DateOffset(months=5)
old=panel[panel['periodo_mes']<cut]; new=panel[panel['periodo_mes']>=cut]
rows=[]
for f in features:
    mu,sd=old[f].mean(),old[f].std(ddof=0)
    rows.append({'feature':f,'mean_hist':mu,'mean_recent':new[f].mean(),'std_shift':(new[f].mean()-mu)/(sd if sd else np.nan)})
drift=pd.DataFrame(rows).sort_values('std_shift',key=lambda s:s.abs(),ascending=False)
drift

## Decision Memory — contrato sugerido
No se escribe automáticamente en producción desde este notebook. La tabla objetivo debe registrar una decisión con trazabilidad y permitir luego comparar resultado observado.

In [ ]:
decision_contract=pd.DataFrame(columns=[
 'decision_id','decision_at','codigo_proyecto','hypothesis','model_version','forecast_horizon_months',
 'recommended_action','approved_action','owner','guardrail','expected_metric','expected_delta',
 'review_at','outcome_metric','outcome_delta','result','learning'
])
decision_contract

## Diseño experimental
Para pricing/descuento: preferir A/B por unidades/segmentos cuando sea comercialmente viable; si no, rollout escalonado por proyecto/periodo y diferencias-en-diferencias.

**Guardrails:** margen, velocidad, tasa de caída, mix, CAC y tiempo de cierre. No optimizar sólo absorción.

## Cadencia operativa
- **Diario:** calidad/refresh.
- **Semanal:** alertas, decisiones y acciones.
- **Mensual:** retraining/challenger + comité macro.
- **Trimestral:** revisión de definición, causalidad, pricing policy y portfolio strategy.

Promover MLflow sólo cuando los datasets, targets y backtests estén validados; primero evidencia, luego registro de experimentos.